In [1]:
print("A")

A


In [2]:
import os
%pwd

'c:\\Users\\Karan\\Desktop\\Kidney-Disease-Deep-Learning-Project\\Kidney-Disease-Classification-Deep-Learning\\research'

In [3]:
os.chdir("../")
%pwd

'c:\\Users\\Karan\\Desktop\\Kidney-Disease-Deep-Learning-Project\\Kidney-Disease-Classification-Deep-Learning'

In [4]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [6]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir = config.root_dir,
            source_URL = config.source_URL,
            local_data_file = config.local_data_file,
            unzip_dir = config.unzip_dir
        )

        return data_ingestion_config



In [9]:
import os
import zipfile
import gdown
from cnnClassifier.utils.common import get_size
from cnnClassifier import logger

In [13]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self) -> str:
        """
        Download file from Google Drive using gdown library.
        """
        try:
            dataset_url = self.config.source_URL
            zip_download_dir = self.config.local_data_file
            os.makedirs("artifacts/data_ingestion", exist_ok=True)
            logger.info(f"Downloading file from {dataset_url} to {zip_download_dir}")
            
            file_id = dataset_url.split("/")[-2]
            prefix = "https://drive.google.com/uc?/expore=download&id="
            gdown.download(prefix + file_id, output=zip_download_dir)
            logger.info(f"File downloaded successfully. Size: {get_size(zip_download_dir)}")

        except Exception as e:
            raise e

    def extract_zip_file(self):
        """
        zip_file_path: str
        Extract the zip file to the specified directory.
        Function returns None
        """

        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)
            

In [17]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-02-11 14:53:25,985: INFO: common]: yaml file: config\config.yaml loaded successfully]
[2026-02-11 14:53:25,987: INFO: common]: yaml file: params.yaml loaded successfully]
[2026-02-11 14:53:25,987: INFO: common]: created directory at: artifacts]
[2026-02-11 14:53:25,990: INFO: common]: created directory at: artifacts/data_ingestion]
[2026-02-11 14:53:25,990: INFO: 1924128960]: Downloading file from https://drive.google.com/file/d/16p6hRANUcAo874dabNr7UaQaxu23pp6n/view?usp=sharing to artifacts/data_ingestion/ct-scan.zip]


Downloading...
From (original): https://drive.google.com/uc?/expore=download&id=16p6hRANUcAo874dabNr7UaQaxu23pp6n
From (redirected): https://drive.google.com/uc?%2Fexpore=download&id=16p6hRANUcAo874dabNr7UaQaxu23pp6n&confirm=t&uuid=cef5f975-b5ca-4150-8503-973b5e907411
To: c:\Users\Karan\Desktop\Kidney-Disease-Deep-Learning-Project\Kidney-Disease-Classification-Deep-Learning\artifacts\data_ingestion\ct-scan.zip
100%|██████████| 1.63G/1.63G [12:24<00:00, 2.19MB/s]


EnsureError: Argument path of type <class 'str'> to <function get_size at 0x000001E16ACA5D80> does not match annotation type <class 'pathlib.Path'>